# SAPC to Pharmacy List Join

Matches SAPC-registered pharmacies to our existing geocoded pharmacy list using fuzzy name and address comparison.

Matching is province-constrained: candidates are only compared within the same province to reduce false positives. The combined score weights name similarity at 60% and address similarity at 40%. A chain guard prevents matching records that belong to different known chains regardless of score.

Three match tiers are produced:
- `auto_match`: score at or above `THRESHOLD_AUTO`, used directly
- `review`: score between thresholds, flagged for manual inspection
- `no_match`: score below `THRESHOLD_REVIEW`, routed to geocoding

Outputs: `geocoded_with_sapc.csv`, `sapc_needs_geocoding.csv`, `needs_human_review.csv`

## Imports and configuration

In [ ]:
import re
import pandas as pd
from rapidfuzz import fuzz, process

GEOCODED_PATH = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results.csv'
SAPC_PATH     = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Source Files\SAPC\sapc_raw.csv'

THRESHOLD_AUTO   = 80   # >= auto match
THRESHOLD_REVIEW = 70   # >= flag for review; < no match

NAME_WEIGHT    = 0.6
ADDRESS_WEIGHT = 0.4

OUT_MATCHED       = 'matched_pharmacies.csv'
OUT_SAPC_ONLY     = 'sapc_only_needs_geocoding.csv'
OUT_GEOCODED_ONLY = 'geocoded_only_review.csv'
OUT_REVIEW        = 'needs_human_review.csv'

print('Config ready.')
print(f'  Auto-match threshold : {THRESHOLD_AUTO}')
print(f'  Review threshold     : {THRESHOLD_REVIEW}')

## Load and inspect datasets

In [ ]:
geo  = pd.read_csv(GEOCODED_PATH, dtype=str)
sapc = pd.read_csv(SAPC_PATH, dtype=str)

print(f'Geocoded : {geo.shape[0]:,} rows x {geo.shape[1]} cols')
print(f'SAPC     : {sapc.shape[0]:,} rows x {sapc.shape[1]} cols')
print()
print('Geocoded columns:', list(geo.columns))
print()
print('SAPC columns:', list(sapc.columns))

## Data preparation

Normalises names and addresses by lowercasing, removing punctuation, expanding abbreviations, and stripping generic pharmacy-related words that reduce discrimination. Province values are normalised to a consistent lowercase form.

Both datasets are deduplicated before matching. The geocoded list deduplicates on `place_id`. The SAPC list deduplicates on Y-number, preferring Active records over Inactive where both exist for the same identifier.

In [ ]:
def clean_text(s):
    if pd.isna(s) or str(s).strip() == '':
        return ''
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # Expand common abbreviations
    s = re.sub(r'\bdr\b', 'doctor', s)
    s = re.sub(r'\bst\b', 'street', s)
    s = re.sub(r'\brd\b', 'road', s)
    s = re.sub(r'\bave\b', 'avenue', s)
    # Remove generic pharmacy words that appear in nearly every name
    generic = [
        r'\bpharmacy\b', r'\bpharmacies\b', r'\bpharm\b',
        r'\bchemist\b',  r'\bapteek\b',     r'\bdispensary\b',
        r'\bdrug\s*store\b', r'\bdrugs\b',  r'\bmedical\b',
        r'\bmedisyne\b', r'\bmedicare\b',   r'\bmedcare\b',
        r'\bclinic\b',
    ]
    for term in generic:
        s = re.sub(term, '', s)
    return re.sub(r'\s+', ' ', s).strip()


def norm_province(s):
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    if 'kwazulu' in s or 'kzn' in s or 'natal' in s:
        return 'kwazulu-natal'
    if 'gauteng' in s:
        return 'gauteng'
    return s


CHAINS = [
    'dis-chem', 'dischem', 'clicks', 'medirite',
    'alpha pharm', 'alphapharm', 'mopani', 'script',
    'wellness warehouse', 'shoprite', 'pick n pay', 'checkers', 'spar',
]

def detect_chain(s):
    s_lower = str(s).lower() if not pd.isna(s) else ''
    for chain in CHAINS:
        if chain in s_lower:
            return chain
    return None


# Prep geocoded — use Google-verified name where available
geo['match_name']       = geo['matched_name'].fillna(geo['practice_name'])
geo['match_addr']       = geo['matched_address'].fillna(geo['address'])
geo['match_name_clean'] = geo['match_name'].apply(clean_text)
geo['match_addr_clean'] = geo['match_addr'].apply(clean_text)
geo['province_norm']    = geo['province'].apply(norm_province)
geo_deduped             = geo.drop_duplicates(subset=['place_id']).copy()
geo_deduped['chain']    = geo_deduped['match_name'].apply(detect_chain)
print(f'Geocoded after dedup on place_id: {len(geo_deduped):,}')

# Prep SAPC — use detail-page name where available, fall back to table Name
sapc['match_name']       = sapc['pharmacy_name'].fillna(sapc['Name'])
sapc['match_addr']       = sapc['street_address'].fillna('')
sapc['match_name_clean'] = sapc['match_name'].apply(clean_text)
sapc['match_addr_clean'] = sapc['match_addr'].apply(clean_text)
sapc['province_norm']    = sapc['Province'].fillna(sapc['province']).apply(norm_province)

# Prefer Active over Inactive for duplicate Y-numbers
status_order = {'active': 0, 'inactive': 1, 'erased': 2}
sapc['status_sort'] = sapc['Status'].fillna('').str.lower().map(lambda x: status_order.get(x, 9))
sapc_deduped = sapc.sort_values('status_sort').drop_duplicates(subset=['Account #']).copy()
sapc_deduped['chain'] = sapc_deduped['match_name'].apply(detect_chain)

print(f'SAPC after dedup on Y-number: {len(sapc_deduped):,}')
print()
print('SAPC province breakdown:')
print(sapc_deduped['province_norm'].value_counts().to_string())
print()
print('SAPC status breakdown:')
print(sapc_deduped['Status'].value_counts().to_string())
print()
print('Chain breakdown (geocoded):')
print(geo_deduped['chain'].value_counts().head(10).to_string())
print()
print('Chain breakdown (SAPC):')
print(sapc_deduped['chain'].value_counts().head(10).to_string())

## Fuzzy matching

For each SAPC pharmacy, scores all candidates in the geocoded list that share the same province. The chain guard short-circuits to 0 for confirmed chain mismatches (e.g. a Clicks record cannot match a Dis-Chem record regardless of name similarity).

In [ ]:
def combined_score(name1, name2, addr1, addr2, chain1=None, chain2=None):
    # Chain guard: two known but different chains cannot match
    if chain1 and chain2 and chain1 != chain2:
        return 0
    name_score = fuzz.token_sort_ratio(name1, name2)
    addr_score = fuzz.token_set_ratio(addr1, addr2)
    return NAME_WEIGHT * name_score + ADDRESS_WEIGHT * addr_score


results = []

# Group geocoded by province for faster candidate lookup
geo_by_province = {prov: grp.reset_index(drop=True)
                   for prov, grp in geo_deduped.groupby('province_norm')}

total = len(sapc_deduped)

for i, sapc_row in sapc_deduped.iterrows():
    prov       = sapc_row['province_norm']
    candidates = geo_by_province.get(prov, geo_deduped)

    if len(candidates) == 0:
        results.append({
            'sapc_y_number':  sapc_row.get('Account #'),
            'sapc_name':      sapc_row['match_name'],
            'sapc_address':   sapc_row['match_addr'],
            'sapc_status':    sapc_row.get('Status'),
            'sapc_province':  sapc_row['province_norm'],
            'match_type':     'no_match',
            'combined_score': 0,
        })
        continue

    scores     = candidates.apply(
        lambda r: combined_score(
            sapc_row['match_name_clean'], r['match_name_clean'],
            sapc_row['match_addr_clean'], r['match_addr_clean'],
            chain1=sapc_row.get('chain'),
            chain2=r.get('chain'),
        ), axis=1
    )
    best_idx   = scores.idxmax()
    best_score = scores[best_idx]
    best_geo   = candidates.loc[best_idx]

    if best_score >= THRESHOLD_AUTO:
        match_type = 'auto_match'
    elif best_score >= THRESHOLD_REVIEW:
        match_type = 'review'
    else:
        match_type = 'no_match'

    results.append({
        'sapc_y_number':          sapc_row.get('Account #'),
        'sapc_name':              sapc_row['match_name'],
        'sapc_address':           sapc_row['match_addr'],
        'sapc_status':            sapc_row.get('Status'),
        'sapc_province':          sapc_row['province_norm'],
        'sapc_owner':             sapc_row.get('owner'),
        'sapc_responsible_pharm': sapc_row.get('responsible_pharmacist'),
        'sapc_licence':           sapc_row.get('licence_number'),
        'sapc_registration_date': sapc_row.get('registration_date'),
        'sapc_inspection':        sapc_row.get('inspection'),
        'sapc_city':              sapc_row.get('city'),
        'match_type':             match_type,
        'combined_score':         round(best_score, 1),
        'name_score':             round(fuzz.token_sort_ratio(
                                      sapc_row['match_name_clean'],
                                      best_geo['match_name_clean']), 1),
        'addr_score':             round(fuzz.token_set_ratio(
                                      sapc_row['match_addr_clean'],
                                      best_geo['match_addr_clean']), 1),
        'geo_practice_id':        best_geo.get('pharmacy_id'),
        'geo_name':               best_geo['match_name'],
        'geo_address':            best_geo['match_addr'],
        'geo_place_id':           best_geo.get('place_id'),
        'geo_lat':                best_geo.get('lat'),
        'geo_lng':                best_geo.get('lng'),
        'geo_province':           best_geo.get('province_norm'),
    })

    if len(results) % 250 == 0:
        print(f'  {len(results):,} / {total:,} processed...')

results_df = pd.DataFrame(results)
print(f'\nMatching complete: {len(results_df):,} SAPC pharmacies processed')
print()
print('Match type breakdown:')
print(results_df['match_type'].value_counts().to_string())

## Inspect match quality

Review the lowest-scoring auto matches before export — these are the records most likely to be incorrect. Adjust `THRESHOLD_AUTO` if too many borderline matches appear here.

In [ ]:
auto     = results_df[results_df['match_type'] == 'auto_match'].sort_values('combined_score')
review   = results_df[results_df['match_type'] == 'review'].sort_values('combined_score', ascending=False)
no_match = results_df[results_df['match_type'] == 'no_match']

print(f'Auto matches  : {len(auto):,}  (score >= {THRESHOLD_AUTO})')
print(f'Review needed : {len(review):,}  (score {THRESHOLD_REVIEW}-{THRESHOLD_AUTO})')
print(f'No match      : {len(no_match):,}  (score < {THRESHOLD_REVIEW})')
print()

cols = ['sapc_name', 'sapc_address', 'geo_name', 'geo_address', 'combined_score']

print('Lowest-confidence auto matches (most likely errors):')
display(auto[cols].head(10))

print()
print('Sample review candidates:')
display(review[cols].head(10))

print()
print('Sample no-matches (will be routed to geocoding):')
display(no_match[['sapc_y_number', 'sapc_name', 'sapc_address', 'sapc_status']].head(10))

## Export

Appends matched SAPC fields to the full geocoded dataframe. Unmatched and review records are exported separately for geocoding.

In [ ]:
SAPC_APPEND_COLS = [
    'geo_place_id',
    'sapc_y_number', 'sapc_name', 'sapc_address', 'sapc_city',
    'sapc_status', 'sapc_owner', 'sapc_responsible_pharm',
    'sapc_licence', 'sapc_registration_date', 'sapc_inspection',
    'combined_score', 'name_score', 'addr_score',
]

# Left join keeps all geocoded records; SAPC columns are NaN where no match
matched = pd.merge(
    geo_deduped,
    results_df[results_df['match_type'] == 'auto_match'][SAPC_APPEND_COLS],
    left_on='place_id',
    right_on='geo_place_id',
    how='left',
)
matched.drop(columns=['geo_place_id'], inplace=True)
matched.to_csv('geocoded_with_sapc.csv', index=False)
print(f'Geocoded with SAPC: {len(matched):,} rows -> geocoded_with_sapc.csv')
print(f'  Matched to SAPC     : {matched["sapc_y_number"].notna().sum():,}')
print(f'  Unmatched geocoded  : {matched["sapc_y_number"].isna().sum():,}')

# Export SAPC records that need geocoding (no_match and review tiers)
needs_geocoding = results_df[results_df['match_type'].isin(['no_match', 'review'])][
    [
        'sapc_y_number', 'sapc_name', 'sapc_address', 'sapc_city',
        'sapc_province', 'sapc_status', 'sapc_owner',
        'sapc_responsible_pharm', 'sapc_licence',
        'sapc_registration_date', 'sapc_inspection',
        'match_type', 'combined_score',
    ]
].copy()
needs_geocoding.to_csv('sapc_needs_geocoding.csv', index=False)
print(f'\nNeeds geocoding: {len(needs_geocoding):,} records -> sapc_needs_geocoding.csv')
print(f'  No match : {(needs_geocoding["match_type"] == "no_match").sum():,}')
print(f'  Review   : {(needs_geocoding["match_type"] == "review").sum():,}')

display(matched.head(3))